# データセット変換
データセット変換リスト.xlsxを読み込み、SASデータセットをDeltaテーブルとして保存する

## 変換リストの読み込み

In [0]:
%pip install -q openpyxl

In [0]:
# 変換リストの読み込み
import pandas as pd

volume_path = "/Volumes/training_bs/user_uenishi_sas/input"
xlsx_path = f"{volume_path}/データセット変換リスト.xlsx"
pdf = pd.read_excel(xlsx_path, engine="openpyxl")
conversion_list = spark.createDataFrame(pdf)
display(conversion_list)

In [0]:
# from pyspark.sql import functions as F
# conversion_list = conversion_list.filter(F.col("テーブル") == "conv_uri_200505")
# display(conversion_list)

## データセットの変換・テーブル保存

In [0]:
import os
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, DoubleType, IntegerType, DateType

success_count = 0
skip_count = 0

# 属性名 → Sparkデータ型のマッピング
type_map = {
    "string": StringType(),
    "double": DoubleType(),
    "int": IntegerType(),
    "date": DateType(),
}

# データセット単位のグループ一覧を取得
group_cols = ["ボリューム", "データセット", "カタログ", "スキーマ", "テーブル"]
groups = conversion_list.select(group_cols).distinct().collect()

for g in groups:
    vol, dataset, catalog, schema, table = g["ボリューム"], g["データセット"], g["カタログ"], g["スキーマ"], g["テーブル"]
    file_path = os.path.join(vol, dataset)
    ext = os.path.splitext(dataset)[1].lower()
    full_table_name = f"{catalog}.{schema}.{table}"

    print(f"[{success_count + skip_count + 1}] {dataset} → {full_table_name}")

    # 存在チェック
    if not os.path.exists(file_path):
        print(f"  ⚠ ファイルが見つかりません: {file_path} - スキップ")
        skip_count += 1
        continue

    # 読み込み (pandas.read_sas で .sas7bdat を読む)
    try:
        pdf_data = pd.read_sas(file_path, format="sas7bdat", encoding="utf-8")
    except Exception:
        # encoding が合わない場合は latin-1 で再試行
        pdf_data = pd.read_sas(file_path, format="sas7bdat", encoding="latin-1")

    df = spark.createDataFrame(pdf_data)

    # スキーマ定義に基づく型変換
    col_mappings = (
        conversion_list
        .filter(
            (F.col("ボリューム") == vol) &
            (F.col("データセット") == dataset) &
            (F.col("テーブル") == table)
        )
        .select("変数名", "属性")
        .collect()
    )

    # col_mappingsに基づいてカラムの型を変換
    for mapping in col_mappings:
        col_name = mapping["変数名"]
        col_type = mapping["属性"]

        if col_name not in df.columns:
            print(f"  ⚠ カラム '{col_name}' がデータに存在しません - スキップ")
            continue

        if col_type in type_map:
            df = df.withColumn(col_name, F.col(col_name).cast(type_map[col_type]))

    # テーブルとして保存
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_table_name)
    print(f"  ✓ 保存完了: {full_table_name} ({df.count()} 行)")
    success_count += 1

print(f"\n完了: 成功 {success_count} 件 / スキップ {skip_count} 件")

## 保存結果の確認

In [0]:
for row in conversion_list.select("カタログ", "スキーマ", "テーブル").distinct().collect():
    full_table_name = f'{row["カタログ"]}.{row["スキーマ"]}.{row["テーブル"]}'
    print(f"--- {full_table_name} ---")
    if not spark.catalog.tableExists(full_table_name):
        print("  ⚠ テーブルが存在しません - スキップ")
        continue
    spark.sql(f"SELECT * FROM {full_table_name} LIMIT 5").display()